# Phần IV — Bài 4: Apriori và FP-Growth (8 giao dịch)

Theo trang 44–45, **min_support_count > 1** (số lần xuất hiện tối thiểu là 2). Yêu cầu **a)** tìm tất cả tập phổ biến bằng Apriori và **b)** bằng FP-Growth. **Cả hai thuật toán được cài đặt riêng trong notebook**, không thay thế FP-Growth bằng Apriori.

**Họ tên:** Nguyễn Lộc  \n**MSSV:** 6451071043  \n**Nguồn đề:** *Tàu liệu LAB1.pdf*, trang 43–45.  \n**Dữ liệu:** nhập trực tiếp từ bảng trong PDF; không cần tải thêm file.

> Chạy **Runtime → Run all** trên Colab. Mỗi notebook độc lập và có lệnh in họ tên/MSSV.

In [1]:
HO_TEN='NGUYỄN LỘC'
MSSV='6451071043'
print(f'Họ tên: {HO_TEN} | MSSV: {MSSV}')

Họ tên: NGUYỄN LỘC | MSSV: 6451071043


In [2]:
from itertools import combinations
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def support_count(transactions, itemset):
    s=set(itemset)
    return sum(s.issubset(set(t)) for t in transactions)

def apriori_manual(transactions, min_support_count):
    """Thuật toán Apriori: join + prune ở mỗi mức k."""
    transactions=[frozenset(t) for t in transactions]
    n=len(transactions)
    all_items=sorted(set().union(*transactions))
    candidates=[frozenset([x]) for x in all_items]
    levels=[]; counts={}
    k=1
    while candidates:
        level={c:support_count(transactions,c) for c in candidates}
        frequent={c:cnt for c,cnt in level.items() if cnt>=min_support_count}
        if not frequent: break
        levels.append(frequent)
        counts.update(frequent)
        prev=set(frequent)
        candidate_next=set()
        for a,b in combinations(sorted(prev,key=lambda z:tuple(sorted(z))),2):
            merged=a|b
            if len(merged)==k+1 and all(frozenset(s) in prev for s in combinations(merged,k)):
                candidate_next.add(merged)
        candidates=sorted(candidate_next,key=lambda x:tuple(sorted(x)))
        k+=1
    return levels,counts

def itemsets_table(counts,n):
    rows=[{'k':len(items),'Tập phổ biến':', '.join(sorted(items)), 'Support count':cnt, 'Support':round(cnt/n,5)} for items,cnt in counts.items()]
    return pd.DataFrame(rows).sort_values(['k','Tập phổ biến']).reset_index(drop=True) if rows else pd.DataFrame(columns=['k','Tập phổ biến','Support count','Support'])

def rules_from_counts(counts, transactions, min_conf, *, target_prefix=None):
    n=len(transactions)
    rows=[]
    for whole,both_count in counts.items():
        if len(whole)<2: continue
        for size in range(1,len(whole)):
            for left_tuple in combinations(sorted(whole),size):
                left=frozenset(left_tuple)
                right=whole-left
                if target_prefix is not None and not (len(right)==1 and next(iter(right)).startswith(target_prefix)):
                    continue
                left_count=support_count(transactions,left)
                right_count=support_count(transactions,right)
                conf=both_count/left_count
                if conf+1e-12>=min_conf:
                    rows.append({'Vế trái':', '.join(sorted(left)), 'Vế phải':', '.join(sorted(right)),
                                 'Support count':both_count,'Support':round(both_count/n,5),
                                 'Confidence':round(conf,5),'Lift':round(conf/(right_count/n),5)})
    return pd.DataFrame(rows).sort_values(['Confidence','Support','Vế trái'],ascending=[False,False,True]).reset_index(drop=True) if rows else pd.DataFrame(columns=['Vế trái','Vế phải','Support count','Support','Confidence','Lift'])

def plot_rules(rules,title):
    if len(rules)==0:
        print('Không có luật nào vượt qua ngưỡng nên không vẽ scatter.')
        return
    fig,ax=plt.subplots(figsize=(7,4.2))
    ax.scatter(rules['Support'],rules['Confidence'],s=65,alpha=.8)
    ax.set(xlabel='Support',ylabel='Confidence',title=title)
    ax.grid(alpha=.3)
    plt.tight_layout();plt.show()


## Dữ liệu và ngưỡng

In [3]:
# Bài 4 — 8 giao dịch A/B/C/D; min_support_count > 1 nghĩa là >= 2
transactions=[
    ['A','B','C','D'],
    ['A','B','C'],
    ['A','B','C'],
    ['A','B','D'],
    ['A','B'],
    ['A','C','D'],
    ['A','D'],
    ['B','C','D']
]
min_count=2
display(pd.DataFrame({'TID':[f'T{i}' for i in range(1,9)],'Items':[', '.join(t) for t in transactions]}))
print('min_support_count > 1 ⇒ support count ít nhất 2 (support >= 25%).')


,TID,Items
0,T1,"A, B, C, D"
1,T2,"A, B, C"
2,T3,"A, B, C"
3,T4,"A, B, D"
4,T5,"A, B"
5,T6,"A, C, D"
6,T7,"A, D"
7,T8,"B, C, D"


min_support_count > 1 ⇒ support count ít nhất 2 (support >= 25%).


## a) Thuật toán Apriori

In [4]:
apr_levels,apr_counts=apriori_manual(transactions,min_count)
print('APRIORI — số tập phổ biến theo mức:')
for k,lev in enumerate(apr_levels,1):
    print('L'+str(k)+':',len(lev),'tập');display(itemsets_table(lev,len(transactions)))


APRIORI — số tập phổ biến theo mức:
L1: 4 tập


,k,Tập phổ biến,Support count,Support
0,1,A,7,0.875
1,1,B,6,0.750
2,1,C,5,0.625
3,1,D,5,0.625


L2: 6 tập


,k,Tập phổ biến,Support count,Support
0,2,"A, B",5,0.625
1,2,"A, C",4,0.500
2,2,"A, D",4,0.500
3,2,"B, C",4,0.500
4,2,"B, D",3,0.375
5,2,"C, D",3,0.375


L3: 4 tập


,k,Tập phổ biến,Support count,Support
0,3,"A, B, C",3,0.375
1,3,"A, B, D",2,0.250
2,3,"A, C, D",2,0.250
3,3,"B, C, D",2,0.250


## b) Thuật toán FP-Growth

In [5]:
class FPNode:
    def __init__(self,item=None,parent=None):
        self.item=item; self.count=0; self.parent=parent; self.children={}; self.link=None

def build_fp_tree(weighted_transactions,min_count):
    item_freq=Counter()
    for transaction,weight in weighted_transactions:
        for item in set(transaction): item_freq[item]+=weight
    frequent={x:c for x,c in item_freq.items() if c>=min_count}
    if not frequent:return None,{}
    root=FPNode()
    header={item:[] for item in frequent}
    for transaction,weight in weighted_transactions:
        ordered=sorted((x for x in set(transaction) if x in frequent),key=lambda x:(-frequent[x],x))
        node=root
        for item in ordered:
            if item not in node.children:
                node.children[item]=FPNode(item,node)
                header[item].append(node.children[item])
            node=node.children[item]
            node.count+=weight
    return root,header

def fpgrowth_manual(transactions,min_count):
    """FP-Growth dùng FP-tree và conditional pattern bases (không gọi Apriori)."""
    result={}
    def mine(weighted,prefix):
        root,header=build_fp_tree(weighted,min_count)
        if root is None:return
        item_support={item:sum(n.count for n in nodes) for item,nodes in header.items()}
        for item in sorted(header,key=lambda x:(item_support[x],x)):
            curr=prefix|frozenset([item]);result[curr]=item_support[item]
            cond=[]
            for node in header[item]:
                path=[];anc=node.parent
                while anc is not None and anc.item is not None:
                    path.append(anc.item);anc=anc.parent
                if path:cond.append((path,node.count))
            if cond:mine(cond,curr)
    mine([(set(t),1) for t in transactions],frozenset())
    return result


In [6]:
fp_counts=fpgrowth_manual(transactions,min_count)
print('FP-GROWTH — kết quả FP-tree/conditional pattern base:')
display(itemsets_table(fp_counts,len(transactions)))
print('Apriori và FP-Growth cho cùng số tập phổ biến:',len(apr_counts),'/',len(fp_counts))
print('Giống nhau hoàn toàn cả tập phổ biến và support count?',apr_counts==fp_counts)
assert apr_counts==fp_counts,'Hai thuật toán chưa nhất quán!'


FP-GROWTH — kết quả FP-tree/conditional pattern base:


,k,Tập phổ biến,Support count,Support
0,1,A,7,0.875
1,1,B,6,0.750
2,1,C,5,0.625
3,1,D,5,0.625
4,2,"A, B",5,0.625
5,2,"A, C",4,0.500
6,2,"A, D",4,0.500
7,2,"B, C",4,0.500
8,2,"B, D",3,0.375
9,2,"C, D",3,0.375


Apriori và FP-Growth cho cùng số tập phổ biến: 14 / 14
Giống nhau hoàn toàn cả tập phổ biến và support count? True
